# Iteration 1 - MLP baseline (MLP_Training)

Simplest net: flatten the image and use dense layers. Purpose: measure the score with zero visual inductive bias. We expect it to UNDERFIT - low train accuracy AND low val accuracy, small gap.

## 1. Setup

In [ ]:
# --- install + imports ---
!pip -q install wandb
import os, math, numpy as np, pandas as pd
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import wandb

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)
torch.manual_seed(42); np.random.seed(42)

## 2. Dataset (Kaggle)

In [ ]:
# --- get the dataset from Kaggle (upload your kaggle.json when prompted) ---
from google.colab import files
files.upload()                                   # >>> select kaggle.json
!mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
!mkdir -p data
!kaggle competitions download -c challenges-in-representation-learning-facial-expression-recognition-challenge -p data
!cd data && unzip -o "*.zip" >/dev/null && echo "unzipped" && ls
# we use icml_face_data.csv: it has the official Training/PublicTest/PrivateTest split

In [ ]:
# --- data: parse the csv, make train/val/test, dataloaders ---
IMG_SIZE, NUM_CLASSES = 48, 7
EMOTIONS = {0:"Angry",1:"Disgust",2:"Fear",3:"Happy",4:"Sad",5:"Surprise",6:"Neutral"}
FER_MEAN, FER_STD = 0.507, 0.255   # train-split pixel stats (scaled to [0,1])

def parse_pixels(s):
    return np.fromstring(s, dtype=np.float32, sep=" ").reshape(IMG_SIZE, IMG_SIZE)

class FERDataset(Dataset):
    def __init__(self, imgs, labels, tf):
        self.imgs, self.labels, self.tf = imgs, labels, tf
    def __len__(self): return len(self.imgs)
    def __getitem__(self, i):
        img = self.tf(self.imgs[i].astype(np.uint8))
        if self.labels is None: return img
        return img, int(self.labels[i])

def build_tf(train, augment, normalize=True):
    t = [transforms.ToPILImage()]
    if train and augment:                          # augmentation = our main anti-overfit tool
        t += [transforms.RandomHorizontalFlip(),
              transforms.RandomRotation(10),
              transforms.RandomResizedCrop(IMG_SIZE, scale=(0.85, 1.0))]
    t += [transforms.ToTensor()]
    if normalize: t += [transforms.Normalize([FER_MEAN], [FER_STD])]
    return transforms.Compose(t)

def load_data(csv_path="data/icml_face_data.csv", batch_size=128, augment=False):
    df = pd.read_csv(csv_path); df.columns = [c.strip() for c in df.columns]
    def split(usage):
        sub = df[df["Usage"] == usage]
        X = np.stack([parse_pixels(p) for p in sub["pixels"].values])
        y = sub["emotion"].values.astype(np.int64)
        return X, y
    Xtr,ytr = split("Training"); Xva,yva = split("PublicTest"); Xte,yte = split("PrivateTest")
    tr = FERDataset(Xtr,ytr, build_tf(True,  augment))
    va = FERDataset(Xva,yva, build_tf(False, False))
    te = FERDataset(Xte,yte, build_tf(False, False))
    return (DataLoader(tr, batch_size, shuffle=True),
            DataLoader(va, batch_size, shuffle=False),
            DataLoader(te, batch_size, shuffle=False))

## 3. Log in to W&B

In [ ]:
# --- log in to Weights & Biases (paste key from https://wandb.ai/authorize) ---
wandb.login()

## 4. Helpers (train / eval / sanity)

In [ ]:
# --- training / evaluation helpers ---
def train_one_epoch(model, loader, opt, crit):
    model.train(); tl=correct=n=0
    for x,y in loader:
        x,y = x.to(device), y.to(device)
        opt.zero_grad(); out = model(x); loss = crit(out,y)
        loss.backward(); opt.step()
        tl += loss.item()*x.size(0); correct += (out.argmax(1)==y).sum().item(); n += x.size(0)
    return tl/n, correct/n

@torch.no_grad()
def evaluate(model, loader, crit):
    model.eval(); tl=correct=n=0; preds=[]; tgts=[]
    for x,y in loader:
        x,y = x.to(device), y.to(device)
        out = model(x); loss = crit(out,y)
        tl += loss.item()*x.size(0); p = out.argmax(1)
        correct += (p==y).sum().item(); n += x.size(0)
        preds += p.cpu().tolist(); tgts += y.cpu().tolist()
    return tl/n, correct/n, preds, tgts

In [ ]:
# --- sanity checks: run these once before training (forward + backward) ---
def sanity_checks(model):
    model = model.to(device)
    x = torch.randn(8,1,IMG_SIZE,IMG_SIZE, device=device)
    out = model(x)
    assert out.shape == (8, NUM_CLASSES), out.shape
    print("forward shape OK:", tuple(out.shape))
    y = torch.randint(0, NUM_CLASSES, (8,), device=device)
    loss = nn.CrossEntropyLoss()(model(x), y)
    print(f"loss@init = {loss.item():.3f}  (expect ~ln(7) = {math.log(7):.3f})")
    model.zero_grad(); loss.backward()
    missing = [n for n,p in model.named_parameters() if p.requires_grad and p.grad is None]
    print("params missing grad:", missing if missing else "none -> backward OK")

## 5. Model + sanity check

In [ ]:
# === Iteration 1: MLP (no spatial prior -> expected UNDERFIT) ===
class MLP(nn.Module):
    """Flatten 48x48 -> 2304 and run dense layers. Throws away all 2-D
    structure, so it cannot learn good facial features: our underfit baseline."""
    def __init__(self, hidden=(512,256), dropout=0.0):
        super().__init__()
        layers, d = [], IMG_SIZE*IMG_SIZE
        for h in hidden:
            layers += [nn.Linear(d, h), nn.ReLU()]
            if dropout > 0: layers.append(nn.Dropout(dropout))
            d = h
        self.net  = nn.Sequential(*layers)
        self.head = nn.Linear(d, NUM_CLASSES)
    def forward(self, x):
        return self.head(self.net(torch.flatten(x, 1)))

sanity_checks(MLP())

## 6. Training routine (W&B logging)

In [ ]:
# --- one experiment = one W&B run (MLflow parity: group == experiment) ---
GROUP   = "MLP_Training"
PROJECT = "fer2013-emotion"

def train_run(run_name, model, epochs, lr, batch_size, augment,
              weight_decay=0.0, optimizer="adam", label_smoothing=0.0, save_path=None):
    tr, va, te = load_data(batch_size=batch_size, augment=augment)
    model = model.to(device)
    n_params = sum(p.numel() for p in model.parameters())

    run = wandb.init(project=PROJECT, group=GROUP, name=run_name, reinit=True,
                     config=dict(model_type=GROUP.replace("_Training",""), config=run_name,
                                 lr=lr, batch_size=batch_size, epochs=epochs, augment=augment,
                                 weight_decay=weight_decay, optimizer=optimizer,
                                 label_smoothing=label_smoothing, n_params=n_params))

    wandb.watch(model, log="all", log_freq=100)   # grad + weight histograms

    if optimizer == "adam":
        opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    else:
        opt = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9, nesterov=True,
                              weight_decay=weight_decay)
    crit = nn.CrossEntropyLoss(label_smoothing=label_smoothing)

    best_val, best_state = 0.0, None
    for ep in range(1, epochs+1):
        trl, tra = train_one_epoch(model, tr, opt, crit)
        val, vaa, vp, vt = evaluate(model, va, crit)
        wandb.log({"epoch":ep, "train/loss":trl, "train/acc":tra,
                   "val/loss":val, "val/acc":vaa,
                   "overfit_gap": tra - vaa,
                   "lr": opt.param_groups[0]["lr"]})
        if vaa > best_val:
            best_val = vaa
            best_state = {k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
        print(f"ep{ep:02d}  train {tra:.3f}  val {vaa:.3f}  gap {tra-vaa:+.3f}")

    if best_state is not None: model.load_state_dict(best_state)
    tel, tea, tp, tt = evaluate(model, te, crit)

    from sklearn.metrics import f1_score, classification_report
    cls = [EMOTIONS[i] for i in range(NUM_CLASSES)]
    wandb.run.summary["best_val_acc"] = best_val
    wandb.run.summary["test_acc"]     = tea
    wandb.run.summary["test_f1"]      = f1_score(tt, tp, average="macro")

    rep = classification_report(tt, tp, target_names=cls, output_dict=True, zero_division=0)
    tbl = wandb.Table(columns=["class", "precision", "recall", "f1", "support"])
    for c in cls:
        r = rep[c]; tbl.add_data(c, r["precision"], r["recall"], r["f1-score"], r["support"])

    wandb.log({"val/confusion_matrix":  wandb.plot.confusion_matrix(y_true=vt, preds=vp, class_names=cls),
               "test/confusion_matrix": wandb.plot.confusion_matrix(y_true=tt, preds=tp, class_names=cls),
               "test/per_class_report": tbl})
    print(f"BEST val_acc={best_val:.4f}  TEST acc={tea:.4f}")
    if save_path is not None:
        torch.save(model.state_dict(), save_path); print("saved ->", save_path)
    wandb.finish()
    return best_val, tea

## 7. Manual hyperparameter tuning

In [ ]:
# --- manual hyperparameter tuning: a few configs, each logged as its own run ---
mlp_configs = [
    dict(run_name="MLP h=[256] lr=1e-3",            hidden=(256,),    dropout=0.0, lr=1e-3),
    dict(run_name="MLP h=[512,256] lr=1e-3",        hidden=(512,256), dropout=0.0, lr=1e-3),
    dict(run_name="MLP h=[512,256] lr=5e-4 do=0.3", hidden=(512,256), dropout=0.3, lr=5e-4),
]
for c in mlp_configs:
    model = MLP(hidden=c["hidden"], dropout=c["dropout"])
    train_run(c["run_name"], model, epochs=30, lr=c["lr"], batch_size=128, augment=False)

## 8. Analysis

Fill in after running. Look for train and val accuracy both low and close (small overfit_gap): the signature of underfitting. Fix: a CNN (iteration 2).

## Automated tuning (W&B Sweep)
The manual loop above tunes this architecture. The automated sweep (W&B chooses
the configs and draws parameter-importance charts) is run on our best
architecture in the DeepCNN notebook.